# 🏆 Vietnamese Sports News Summarization — Full Pipeline v2
## Data Augmentation → Fine-tune BARTpho & ViT5 → Evaluation → Demo

**Constraints enforced throughout:**
- ✅ 100% number preservation from source article
- ✅ Summary length 20–25% of original (by word count)

**What's covered:**
| # | Section |
|---|---|
| 1 | Install Dependencies |
| 2 | Imports & Constants |
| 3 | Data Loading, **Augmentation** & Train/Val/Test Split |
| 4 | Dataset & Tokenization |
| 5 | Model 1 — BARTpho-word (LoRA) |
| 6 | Model 2 — ViT5-base (LoRA) |
| 7 | Post-processing: Number Preservation |
| 8 | Inference on Test Set |
| 9 | Evaluation: ROUGE, BLEU, METEOR, BERTScore, Number Accuracy, Length Compliance, **LLM-as-Judge** |
| 10 | Visual Comparison (Matplotlib + **Plotly Interactive**) |
| 11 | Error Analysis |
| 12 | **Model Saving, Zipping & Download** |
| 13 | **Gradio Demo Interface** |

## Section 1 — Install Dependencies

In [1]:
!pip install -q "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.0" \
             datasets evaluate rouge_score bert_score \
             sacrebleu nltk sentencepiece \
             plotly gradio anthropic scikit-learn

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print('Dependencies installed ✓')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 9.4 MB/s eta 0:00:00
Dependencies installed ✓


## Section 2 — Imports & Constants

In [2]:
import os, re, json, shutil, warnings, random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, set_seed,
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate

import matplotlib.pyplot as plt
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

# ── Paths ──────────────────────────────────────────────────────
DATA_FILE       = 'data.csv'   # ← update to your file
OUTPUT_DIR      = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
BARTPHO_OUT_DIR = str(OUTPUT_DIR / 'bartpho_finetuned')
VIT5_OUT_DIR    = str(OUTPUT_DIR / 'vit5_finetuned')
VIT5_PREFIX     = 'tóm tắt: '

# ── Number Regex ───────────────────────────────────────────────
NUMBER_PATTERN = re.compile(
    r'\b\d{4}-\d{4}\b'
    r'|\b\d{1,2}/\d{1,2}/\d{2,4}\b'
    r'|\b\d{1,2}/\d{1,2}\b'
    r'|\b\d+(?:[.,]\d+)*%?\b'
    r'|\b\d+\s*-\s*\d+\b'
    r'|\b\d+[hH]\d{2}\b'
)

def extract_numbers(text: str) -> set:
    return set(NUMBER_PATTERN.findall(str(text)))

def count_words(text: str) -> int:
    return len(str(text).split())

print('Imports done ✓')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Device : cuda
GPU    : Tesla T4
Imports done ✓


## Section 3 — Data Loading, Augmentation & Split

### Data Augmentation Strategy
To enlarge the training set without violating the number-preservation constraint we apply **two lightweight, number-safe techniques on the article only** (summary is never modified):

| Method | What it does | Why it's safe |
|---|---|---|
| **Synonym Swap** | Replaces common Vietnamese sports words with synonyms from a curated dict | Only non-numeric tokens are swapped |
| **Sentence Shuffle** | Randomly reorders sentences in the article | Numbers stay; model can't rely on position |

> Val and Test sets are **never augmented** to keep evaluation clean.

In [3]:
VI_SYNONYMS = {
    # ── TEAM / CLUB ───────────────────────────────────────────
    'câu lạc bộ'        : ['đội bóng', 'CLB'],
    'đội bóng'          : ['câu lạc bộ', 'đội'],
    'đội tuyển'         : ['tuyển', 'đội tuyển quốc gia'],
    'tuyển thủ'         : ['cầu thủ', 'thành viên đội tuyển'],

    # ── MATCH / GAME ──────────────────────────────────────────
    'trận đấu'          : ['cuộc đấu', 'trận'],
    'trận'              : ['trận đấu', 'cuộc đấu'],
    'trận lượt đi'      : ['lượt đi'],
    'trận lượt về'      : ['lượt về'],
    'trận chung kết'    : ['chung kết'],
    'trận bán kết'      : ['bán kết'],
    'trận tứ kết'       : ['tứ kết'],
    'vòng bảng'         : ['giai đoạn vòng bảng'],

    # ── SCORING ───────────────────────────────────────────────
    'ghi bàn'           : ['lập công', 'ghi bàn thắng'],
    'bàn thắng'         : ['bàn ghi được', 'bàn'],
    'bàn thua'          : ['bàn thủng lưới'],
    'lập công'          : ['ghi bàn'],
    'ấn định tỷ số'     : ['ghi bàn quyết định', 'chốt tỷ số'],
    'rút ngắn tỷ số'    : ['thu hẹp cách biệt'],
    'tỷ số'             : ['cách biệt', 'tỷ số trận đấu'],
    'hòa'               : ['cầm hòa', 'chia điểm'],
    'thủng lưới'        : ['để thủng lưới', 'để thua'],

    # ── RESULT / PERFORMANCE ──────────────────────────────────
    'chiến thắng'       : ['thắng lợi', 'giành chiến thắng'],
    'giành chiến thắng' : ['giành thắng lợi', 'thắng'],
    'thất bại'          : ['thua cuộc', 'chịu thất bại'],
    'thắng lợi'         : ['chiến thắng'],
    'vô địch'           : ['giành chức vô địch', 'đoạt chức vô địch'],
    'á quân'            : ['hạng nhì', 'đội về nhì'],

    # ── PLAYERS / POSITIONS ───────────────────────────────────
    'cầu thủ'           : ['tuyển thủ'],
    'thủ môn'           : ['người gác đền', 'cầu thủ bắt gôn'],
    'hậu vệ'            : ['cầu thủ phòng ngự'],
    'tiền vệ'           : ['cầu thủ tuyến giữa'],
    'tiền đạo'          : ['cầu thủ tấn công'],
    'đội trưởng'        : ['thủ quân'],
    'cầu thủ dự bị'     : ['người dự bị', 'cầu thủ vào sân thay người'],

    # ── COACHING STAFF ────────────────────────────────────────
    'huấn luyện viên'   : ['HLV'],
    'HLV trưởng'        : ['huấn luyện viên trưởng'],
    'ban huấn luyện'    : ['ban kỹ thuật', 'ban HLV'],
    'trợ lý huấn luyện viên': ['trợ lý HLV'],

    # ── TOURNAMENT / COMPETITION ──────────────────────────────
    'giải đấu'          : ['giải', 'giải vô địch'],
    'giải vô địch'      : ['giải', 'giải đấu'],
    'mùa giải'          : ['mùa bóng'],
    'vòng loại'         : ['giai đoạn loại'],
    'vòng chung kết'    : ['giai đoạn chung kết'],
    'lượt trận'         : ['vòng đấu'],

    # ── STANDINGS / RANKINGS ──────────────────────────────────
    'bảng xếp hạng'     : ['thứ hạng', 'bảng tổng sắp'],
    'đứng đầu'          : ['dẫn đầu', 'ở vị trí đầu bảng'],
    'dẫn đầu bảng'      : ['đứng đầu bảng xếp hạng'],
    'điểm số'           : ['điểm'],
    'phong độ'          : ['form thi đấu'],

    # ── MATCH ACTIONS ─────────────────────────────────────────
    'sút phạt'          : ['đá phạt'],
    'đá phạt'           : ['sút phạt'],
    'phạt góc'          : ['đá phạt góc', 'corner'],
    'phạt đền'          : ['đá phạt đền', 'penalty'],
    'thẻ vàng'          : ['nhận thẻ vàng', 'bị rút thẻ vàng'],
    'thẻ đỏ'            : ['nhận thẻ đỏ', 'bị đuổi khỏi sân'],
    'kiến tạo'          : ['đường kiến tạo', 'góp công'],
    'vào sân thay người': ['được tung vào sân', 'thay người'],
    'rời sân'           : ['bị thay ra', 'ra sân'],
    'cơ hội'            : ['tình huống nguy hiểm', 'pha bóng nguy hiểm'],

    # ── VENUE ─────────────────────────────────────────────────
    'sân nhà'           : ['sân vận động sân nhà', 'trên sân nhà'],
    'sân khách'         : ['trên sân khách', 'sân đối phương'],
    'sân vận động'      : ['SVĐ', 'sân bóng'],

    # ── GENERAL SPORTS TERMS ──────────────────────────────────
    'thi đấu'           : ['ra sân', 'tham dự'],
    'kết thúc'          : ['khép lại', 'kết thúc trận đấu'],
    'diễn ra'           : ['được tổ chức', 'diễn ra tại'],
    'chứng kiến'        : ['theo dõi', 'quan sát'],
}

def synonym_swap(text: str, prob: float = 0.15, seed: int = 42) -> str:
    rng = random.Random(seed)
    # Sort keys longest-first so multi-word phrases match before single words
    sorted_keys = sorted(VI_SYNONYMS.keys(), key=lambda k: len(k.split()), reverse=True)
    result = text
    for key in sorted_keys:
        if key in result and rng.random() < prob:
            result = result.replace(key, rng.choice(VI_SYNONYMS[key]), 1)
    return result

def sentence_shuffle(text: str, seed: int = 42) -> str:
    rng = random.Random(seed)
    # Split on period followed by whitespace only — avoids breaking 45.000 or 3.1
    sentences = [s.strip() for s in re.split(r'\.\s+', text) if s.strip()]
    if len(sentences) <= 1:
        return text   # nothing to shuffle
    rng.shuffle(sentences)
    return '. '.join(sentences) + '.'

def augment_dataframe(df: pd.DataFrame, aug_factor: int = 2) -> pd.DataFrame:
    augmented_rows = []
    for seed_offset, (_, row) in enumerate(df.iterrows()):
        for aug_idx in range(aug_factor):
            seed = seed_offset * 100 + aug_idx
            new_row = row.copy()
            if aug_idx % 2 == 0:
                new_row['article'] = synonym_swap(str(row['article']), seed=seed)
            else:
                new_row['article'] = sentence_shuffle(str(row['article']), seed=seed)
            new_row['augmented'] = True
            augmented_rows.append(new_row)
    aug_df = pd.DataFrame(augmented_rows)
    df = df.copy()
    df['augmented'] = False
    combined = pd.concat([df, aug_df], ignore_index=True)
    print(f'  Original : {len(df)} | Augmented: {len(aug_df)} | Total: {len(combined)}')
    return combined

In [4]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_FILE)
df['valid'] = df['valid'].astype(str).str.lower() == 'true'

valid_df = df[df['valid'] == True].copy().reset_index(drop=True)
print(f'Total rows : {len(df)} | Valid: {len(valid_df)} | Invalid: {len(df)-len(valid_df)}')
print(valid_df['article_words'].describe().round(1))

train_df, temp_df = train_test_split(valid_df, test_size=0.20, random_state=42)
val_df,   test_df = train_test_split(temp_df,  test_size=0.50, random_state=42)
print(f'\nBefore aug — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

print('\nAugmenting training set (×2)…')
train_df_aug = augment_dataframe(train_df, aug_factor=2)

train_df_aug.to_csv(OUTPUT_DIR / 'train.csv', index=False, encoding='utf-8-sig')
val_df.to_csv(OUTPUT_DIR      / 'val.csv',   index=False, encoding='utf-8-sig')
test_df.to_csv(OUTPUT_DIR     / 'test.csv',  index=False, encoding='utf-8-sig')
print('Splits saved ✓  —  NOTE: manually review test.csv before final evaluation.')

Total rows : 1064 | Valid: 1064 | Invalid: 0
count    1064.0
mean      455.5
std        95.6
min       300.0
25%       381.8
50%       443.0
75%       515.0
max       698.0
Name: article_words, dtype: float64

Before aug — Train: 851 | Val: 106 | Test: 107

Augmenting training set (×2)…
  Original : 851 | Augmented: 1702 | Total: 2553
Splits saved ✓  —  NOTE: manually review test.csv before final evaluation.
